# 02 — Amazon Reviews Sentiment Analysis

**Objective:** Convert review text into positive/negative sentiment labels and compare TF-IDF + Logistic Regression, Multinomial Naive Bayes, and Linear SVM.

## Expected input
Place your authorized review file at `data/amazon_reviews/reviews.csv`. The notebook supports:
- Rating column: `Score` or `overall`
- Text column: `Text` or `reviewText`

Binary-label policy used here: ratings 1–2 = negative, 4–5 = positive, rating 3 excluded. Verify this matches your project definition.

In [ ]:
# 1. Imports
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
DATA_PATH = Path("data/amazon_reviews/reviews.csv")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
# 2. Load and validate data
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Place reviews.csv at {DATA_PATH.resolve()}")
raw = pd.read_csv(DATA_PATH)

rating_col = next((c for c in ["Score", "overall"] if c in raw.columns), None)
text_col = next((c for c in ["Text", "reviewText"] if c in raw.columns), None)
if rating_col is None or text_col is None:
    raise ValueError(f"Could not identify rating/text columns. Found: {list(raw.columns)}")

df = raw[[rating_col, text_col]].dropna().copy()
df[rating_col] = pd.to_numeric(df[rating_col], errors="coerce")
df = df.dropna(subset=[rating_col])
df = df[df[rating_col].isin([1, 2, 3, 4, 5])]
df = df[df[rating_col] != 3].copy()
df["sentiment"] = (df[rating_col] >= 4).astype(int)
df[text_col] = df[text_col].astype(str).str.strip()
df = df[df[text_col].str.len() > 0]

print("Usable reviews:", len(df))
display(df[["sentiment", text_col]].head())
display(df["sentiment"].value_counts().rename(index={0:"negative", 1:"positive"}))

In [ ]:
# 3. Optional sampling for laptop/Colab memory limits
MAX_REVIEWS = None  # e.g. set to 100_000 for a smaller reproducible experiment
if MAX_REVIEWS is not None and len(df) > MAX_REVIEWS:
    df = df.groupby("sentiment", group_keys=False).apply(
        lambda part: part.sample(
            n=min(len(part), int(MAX_REVIEWS * len(part) / len(df))),
            random_state=RANDOM_STATE
        )
    ).reset_index(drop=True)

X = df[text_col].astype(str)
y = df["sentiment"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", len(X_train), "Test:", len(X_test))

In [ ]:
# 4. Build TF-IDF pipelines and train classifiers
MAX_FEATURES = 20_000
models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=MAX_FEATURES, stop_words="english", ngram_range=(1, 2), min_df=2)),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]),
    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=MAX_FEATURES, stop_words="english", ngram_range=(1, 2), min_df=2)),
        ("model", MultinomialNB())
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=MAX_FEATURES, stop_words="english", ngram_range=(1, 2), min_df=2)),
        ("model", LinearSVC(class_weight="balanced"))
    ])
}

fitted = {}
records = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metrics = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    }
    fitted[name] = model
    print(f"\n{name}\n", classification_report(y_test, pred, target_names=["negative", "positive"], zero_division=0))
    for metric, value in metrics.items():
        records.append({"Dataset": "Amazon Reviews", "Model": name, "Metric": metric, "Value": float(value)})

In [ ]:
# 5. Compare results
amazon_results = pd.DataFrame(records)
display(amazon_results.pivot(index="Model", columns="Metric", values="Value"))
amazon_results.to_csv(RESULTS_DIR / "amazon_model_results.csv", index=False)

best_name = amazon_results.query("Metric == 'F1'").sort_values("Value", ascending=False)["Model"].iloc[0]
best_model = fitted[best_name]
print("Best F1 model on this holdout:", best_name)

In [ ]:
# 6. Confusion matrix for selected model
best_pred = best_model.predict(X_test)
ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, display_labels=["Negative", "Positive"], cmap="Blues"
)
plt.title(f"Amazon Reviews — {best_name}")
plt.show()

In [ ]:
# 7. Try predictions on custom reviews
sample_reviews = [
    "The product quality is excellent and delivery was quick.",
    "Very disappointing experience. The item stopped working."
]
sample_predictions = best_model.predict(sample_reviews)
for review, label in zip(sample_reviews, sample_predictions):
    print(("Positive" if label == 1 else "Negative"), "—", review)

## Interpretation checklist

- State that neutral 3-star reviews were excluded from binary classification.
- Report class distribution and whether the split was stratified.
- Compare F1 alongside accuracy, especially if classes are imbalanced.
- Inspect false positives and false negatives before making customer-service recommendations.
- Results are valid only for the dataset version and split used in this run.